# Editeur interactif de cartes

## Dépendances

In [1]:
import ipywidgets as widgets
from ipywidgets import interactive
from IPython.display import display
import csv  
import os
from IPython.display import clear_output

## Fonctions utilitaires

In [2]:
#Définition de index pour les fonctions suivantes
index = 0

# retourne un ensemble de 4 sliders verticaux
def sliderBox(values): 
    ls = [None]*4
    
    for i in range(len(ls)):
        ls[i] = widgets.IntSlider(
            value=values[i],
            min=-100,
            max=100,
            step=5,
            orientation='vertical',
            readout=True,
            readout_format='d')
        
    return widgets.HBox(ls)

# Prend une liste de string et retourne les mêmes objets en int
def str2list(_l):
    l = [0]*len(_l)    
    for i in range(0,len(_l)):
        l[i] = int(_l[i])
    return l

# Traduit une rangée brut en rangée exploitable
def row2expl(row):
    #test1 = default,default,default.png,default text,default ltext,0,0,0,0,default lback,default rtext,0,0,0,0,Card,default rback
    #test1 index = 0, 1, 2, 3, 4, [5,6,7,8] 
    data = row
    lData = str2list(data[5:9])
    rData = str2list(data[11:15])
    data[5:9] = lData
    data[11:15] = rData
    return data

# Effectue le nettoyage du fichier csv obtenu
def csvFileCleaning(_csvFile):
    csvHeader = _csvFile[0]
    csvData = [0]*(len(_csvFile)-1)
    dataToClean = csvFile[1:(len(_csvFile))]
    for j in range(0, len(dataToClean)):
        csvData[j] = row2expl(dataToClean[j])
    return csvHeader, csvData

# Retourne les valeurs des 8 sliders
def fetchSliderValues(_items):
    leftValues = [0]*4
    rightValues = [0]*4
    for i in range(0, 4): #items[3] is left slider box
        leftValues[i] = _items[3].children[i].value
    for j in range(0,4):
        rightValues[j] = _items[5].children[j].value
    return leftValues, rightValues    

# Retourne les textes ecrits
def fetchTextValues(_items):
    cardtext = _items[4].value
    ltext = _items[6].value
    rtext = _items[8].value
    return cardtext, ltext, rtext

# Retourne le png choisi dans le champ
def fetchPNG(_items):
    return _items[13].value

# Retourne le type de Card séléctionné
def fetchClass(_items):
    return _items[12].value

def fetchLine(_items):
    return _items[12].value

# Retourne le nom de la carte
def fetchTitle(_items):
    return _items[1].value

def fetchLeftBack(_items):
    return _items[9].value

def fetchRightBack(_items):
    return _items[11].value

def characterName(_items):
    return _items[10].value

# Inscrit les valeurs actuelles dans csvNewData
def writeChanges(index, _items):
    cardtext, ltext, rtext = fetchTextValues(_items)
    lValues, rValues = fetchSliderValues(_items)
    csvNewData[index-1] = [fetchTitle(_items),
                        characterName(_items),
                        fetchPNG(_items),
                        cardtext,
                        ltext,
                        lValues[0],
                        lValues[1],
                        lValues[2],
                        lValues[3],
                        fetchLeftBack(_items),
                        rtext,
                        rValues[0],
                        rValues[1],
                        rValues[2],
                        rValues[3],
                        fetchClass(_items),
                        fetchRightBack(_items)
                        ]

#Inscrit dans le fichier csv les changements
def saveData():
    with open("utils/cards_data.csv", 'w') as csvfile:  
        # creating a csv writer object  
        csvwriter = csv.writer(csvfile, delimiter = ';')  
        
        # writing the fields  
        csvwriter.writerow(csvHeader)  
        
        # writing the data rows  
        csvwriter.writerows(csvNewData) 

# Charge un png depuis le dossier png 
def loadFile(filename):
    file = open('png/'+filename, "rb")
    image =file.read()
    ImageWidget = widgets.Image(
        value=image,
        format='png',
        width=300,
        height=300,
        )
    return ImageWidget

## Chargement du fichier

In [3]:
with open("utils/cards_data.csv", mode ='r') as file:  
      
    # reading the CSV file  
    csvReader = csv.reader(file, delimiter=';')  
    csvFile = []
    for row in csvReader:
        csvFile.append(row)

### Ordonnement
csvHeader, csvData = csvFileCleaning(csvFile)


csvNewData = csvData
print(csvData)

[['PersoFemininTest', 'Perso Feminin Test', 'rnd1_ttsicon_00008_.png', 'Carte Test 1', 'perd 10 pour chaque', -10, -10, -10, -10, 'Affichage lback masc', 'gagne 10 pour chaque', 10, 10, 10, 10, 'Card', 'Affichage rback masc'], ['PersoMasculinTest', 'Perso Masculin Test', 'rnd1_ttsicon_00004_.png', 'Carte Test 2', 'gagne 20 pour chaque', 20, 20, 20, 20, 'affichage lback fem', 'perd 20 pour chaque', -20, -20, -20, -20, 'Card', 'Affichage rback fem'], ['elecUsines', 'default', 'default.png', 'Il nous faut une nouvelle centrale électrique pour les usines !', 'Centrale à charbon', 25, -10, -10, 0, 'La mise en place de la centrale a pollué les environs', 'Centrale nucléaire', 25, 0, -10, -10, 'Card', 'La construction a entraîné des coûts supplémentaires'], ['manqueDeGaz', 'default', 'default.png', "Il n'y a plus assez de gaz pour des usines !", 'Fermer les centrales', -15, 15, 0, 0, 'None', 'Importer du gaz étranger', 15, 0, 0, -15, 'Card', "L'import de gaz donne des envies aux industriels"]

## Création des items graphiques

In [4]:
# Crée une liste d'items graphiques, ré-appelable
def createItems(index):
    # Sliders
    leftAction = sliderBox(csvNewData[index][5:9])
    rightAction = sliderBox(csvNewData[index][11:15])
    
    # Texte de la carte
    textCard = widgets.Textarea(
            value=csvNewData[index][3],
            placeholder='Texte description de carte',
            disabled=False)
    
    # Texte quand le champ de gauche est selectionné
    textLeft = widgets.Textarea(
            value=csvNewData[index][4],
            placeholder='Texte description de carte',
            disabled=False)
    
    # Texte quand le champ de droit est selectionné
    textRight = widgets.Textarea(
            value=csvNewData[index][10],
            placeholder='Texte description de carte',
            disabled=False)

    #Nom fonctionnel de la carte
    Title = widgets.Text(
        value=csvNewData[index][0],
        placeholder='Titre dictionnaire de la carte',
        disabled=False   
    )

    items_layout = widgets.Layout( width='auto')

    sliderIcons = widgets.HBox([widgets.Label(value="Industrie     |"),
                         widgets.Label(value="Ecologie    |"),
                         widgets.Label(value="Population    |"),
                         widgets.Label(value="$$$")],layout = items_layout)

    loadButton = widgets.Button(description="Charger image")

    #Selectionne le type de la carte
    classToggle = widgets.ToggleButtons(
        options=['Card', 'Subclass Card'],
        disabled=False,
        button_style='', # 'success', 'info', 'warning', 'danger' or ''
        tooltips=['Classe Card régulière', 'Subclass de Card pour multiples sweep'],
    #     icons=['check'] * 3
    )

    # Type de voix (non implémenté en l'absence de système de son)
    soundChoice = widgets.SelectMultiple(
        options=['Grave','Grave-Mezzo', 'Mezzo', 'Mezzo-Aigu', 'Aigu'],
        value=['Mezzo'],
        #rows=10,
        description='Caractère de la voix',
        disabled=False
    )
    

    # Choix du fichier parmi ceux mis dans le dossier
    fileLocation = widgets.Dropdown(
        options=os.listdir('png'),
        value=str(csvNewData[index][2]),
        description='png:',
        disabled=False,
        )
    
    # Texte quand le champ de gauche est swipé, optionnel
    LeftBack = widgets.Textarea(
            value=csvNewData[index][9],
            placeholder='Texte arrière choix gauche',
            disabled=False)
    
    # Texte quand le champ de droit est swipé, optionnel
    RightBack = widgets.Textarea(
            value=csvNewData[index][16],
            placeholder='Texte arrière choix droit',
            disabled=False)
    
    CharacterName = widgets.Textarea(
            value=csvNewData[index][1],
            placeholder='Nom du personnage',
            disabled=False)

    saveButton = widgets.Button(description="Sauvegarder")
    
    ImageWidget = loadFile(filename=csvNewData[index][2])
    
    # initie la liste d'items
    items = [sliderIcons, #0
         Title, #1
         sliderIcons, #2
         leftAction, #3
         textCard, #4
         rightAction, #5
         textLeft, #6
         ImageWidget, #7
         textRight, #8
         LeftBack, #9
         CharacterName, #10
         RightBack, #11
         classToggle, #12
         fileLocation, #13
         soundChoice] #14
    return items

### Fonction pour l'affichage des items
def show(_items):
    display(widgets.GridBox(_items, layout=widgets.Layout(grid_template_columns="repeat(3, 310px)")))
    #Affiche les _items mis en appel dans une répartition de 3 items/ligne , chacun séparés par 310 px

## Lancement de l'éditeur de carte

In [5]:
### Définitions

index = 0 #Redéfinition de l'index

#Champ de texte pour la ligne
int_range = widgets.BoundedIntText(
    value=index+1,
    min=1,
    max=300,
    step=1,
    description='Carte N°',
    disabled=False
)

button = widgets.Button(description='Sauvegarder')
#Organisation
box = widgets.HBox([int_range, button])
output2 = widgets.Output()

### Affichage de lancement
display(box, output2)
mainItems = createItems(index)
show(mainItems)

### Interrupt based events
#fonction bouton
def on_button_clicked(b):
    writeChanges(index+1, mainItems) # ecrit les changements sur csvNewData
    saveData() #Ecrit les changements sur cards_data.csv

#fonction champ de numéro
def on_value_change(change):
    global mainItems, testItems, index #globaux pour les fetch
    
    if (mainItems != None): #Sauvegarde automatique
        writeChanges(index+1, mainItems)

    clear_output() #Nettoie l'affichage
    index = int_range.value-1
    mainItems = createItems(index) #Actualise les items selon leurs valeurs dans la data
    display(box, output2) #Affichage des outils
    show(mainItems) #Affichage de la grille
    
def on_png_change(change):
    global mainItems, csvNewData
    writeChanges(index, mainItems)
    clear_output()
    csvNewData[index][1] = fetchPNG(mainItems)
    loadFile(fetchPNG(mainItems))
    mainItems = createItems(index) #Actualise les items selon leurs valeurs dans la data
    display(box, output2) #Affichage des outils
    show(mainItems) #Affichage de la grille

#interrupts
int_range.observe(on_value_change, names='value')
#mainItems[10].observe(on_value_change, names='value')
button.on_click(on_button_clicked)

Output()

GridBox(children=(HBox(children=(Label(value='Industrie     |'), Label(value='Ecologie    |'), Label(value='Po…